In [27]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root added:", PROJECT_ROOT)

Project root added: c:\Users\Bluepal\Desktop\AI_RESEARCH_AGENT


# AI Research Agent — Exploration Notebook

**Purpose:**  
This notebook demonstrates the internal behavior of the AI Research Agent system.  
It runs the full pipeline once and inspects:

- Research planning
- Iterative discovery
- Vector memory reuse vs web ingestion
- Agreement reasoning
- Summary scoring
- Final evaluation

This notebook is for **system understanding**, not benchmarking.

In [28]:
from src.controller.run import run_pipeline
from src.vector_store.client import VectorStoreClient
from src.trace.research_trace import ResearchTrace

import pandas as pd
import textwrap

## 1️⃣ Define Query & Mode

In [29]:
query = "Impact of GEN AI on software engineering jobs"
mode = "standard"   # quick | standard | deep

## 2️⃣ Run Full Research Pipeline

In [30]:
vector_client = VectorStoreClient(embedding_dim=384)
trace = ResearchTrace()

summaries, trace_text, report_text, pdf_path, evaluation = run_pipeline(
    user_query=query,
    mode=mode,
    vector_client=vector_client,
    trace=trace,
)

print("Pipeline complete.")

Pipeline complete.


## 3️⃣ Research Plan

In [31]:
plan_section = trace_text.split("DISCOVERY ITERATION")[0]
print(plan_section)


RESEARCH PLAN
Goal:
Assess the multifaceted impact of generative AI on software engineering roles

Dimensions:
- Job Role Transformation
- Skillset Evolution Requirements
- Productivity and Workflow Changes
- Industry Adoption and Implementation Trends
- Ethical and Societal Implications
- Economic and Labor Market Shifts




## 4️⃣ Discovery Iterations Overview

In [32]:
for line in trace_text.splitlines():
    if "DISCOVERY ITERATION" in line:
        print(line)

DISCOVERY ITERATION 1
DISCOVERY ITERATION 2


## 5️⃣ Vector Reuse vs Web Ingestion
Shows when the system reused memory vs searched the web.

In [33]:
for line in trace_text.splitlines():
    if "Reused summary" in line or "WEB INGESTION" in line:
        print(line)

Reused summary S1 from vector DB (URL: https://www.mckinsey.com/capabilities/tech-and-ai/our-insights/the-economic-potential-of-generative-ai-the-next-productivity-frontier)
Reused summary S2 from vector DB (URL: https://www.ness.com/insights/generative-ai-improves-software-engineering-productivity-by-70-says-ness-zinnov-study)
Reused summary S3 from vector DB (URL: https://www.dhs.gov/sites/default/files/2025-01/2024_1219_impact_of_genai_on_software_engineering_activities_minkiewicz.pdf)
WEB INGESTION — Q2


## 6️⃣ Agreement Map (Cross-Source Reasoning)

In [34]:
start = trace_text.find("AGREEMENT MAP")
end = trace_text.find("TOTAL SCORES")
print(trace_text[start:end])

AGREEMENT MAP
S1 → S2: strongly_supports
S1 → S3: strongly_supports
S1 → S4: partially_supports
S2 → S1: strongly_supports
S2 → S3: strongly_supports
S2 → S4: partially_supports
S3 → S1: strongly_supports
S3 → S2: strongly_supports
S3 → S4: partially_supports
S4 → S1: partially_supports
S4 → S2: partially_supports
S4 → S3: partially_supports




## 7️⃣ Summary Scoring Table

In [35]:
df = pd.DataFrame([
    {
        "ID": s["id"],
        "Credibility": s.get("credibility_score", 0),
        "Agreement": s.get("agreement_score", 0),
        "Total Score": s.get("total_score", 0),
        "Domain": s.get("domain"),
        "URL": s.get("url"),
    }
    for s in summaries
])

df

,ID,Credibility,Agreement,Total Score,Domain,URL
0,S1,0,13,13,mckinsey.com,https://www.mckinsey.com/capabilities/tech-and...
1,S2,1,13,14,ness.com,https://www.ness.com/insights/generative-ai-im...
2,S3,2,13,15,dhs.gov,https://www.dhs.gov/sites/default/files/2025-0...
3,S4,7,9,16,arxiv.org,https://arxiv.org/pdf/2508.16811


## 8️⃣ Final Evidence Summaries

In [36]:
for s in summaries:
    print(f"\n{s['id']} (Score={s.get('total_score')}):")
    print(textwrap.fill(s["summary"], width=100))


S1 (Score=13):
Generative AI is poised to unleash the next wave of productivity, with potential impacts on the
workforce and trillions of dollars in value across sectors. Breakthroughs in foundation models, part
of deep learning, have enabled new capabilities and vastly improved existing ones across various
modalities, including images, video, audio, and computer code. These models can process large and
varied sets of unstructured data and perform multiple tasks, such as classification, editing,
summarization, answering questions, and drafting new content. The speed of generative AI technology
development is rapid, with advancements in models like ChatGPT, GPT-4, Claude, and PaLM 2. Our
research estimates that generative AI could add $2.6 trillion to $4.4 trillion annually across 63
use cases, increasing the impact of all artificial intelligence by 15 to 40 percent. The value of
generative AI use cases falls across four areas: customer operations, marketing and sales, software
enginee

## 9️⃣ Generated Report Preview (First 1200 Characters)

In [37]:
print(report_text[:1200])

@@TITLE@@
Impact of Generative Artificial Intelligence on Software Engineering Jobs
@@TITLE@@

@@Executive Summary@@
Generative Artificial Intelligence (GenAI) is poised to significantly impact the workforce, with potential productivity gains and trillions of dollars in value across sectors [S1]. Breakthroughs in foundation models have enabled new capabilities and vastly improved existing ones across various modalities, including images, video, audio, and computer code [S1]. GenAI has the potential to automate 60 to 70 percent of employees' time today, changing the anatomy of work and augmenting individual workers by automating some of their activities [S1]. The technology could add $2.6 trillion to $4.4 trillion annually across 63 use cases, increasing the impact of all artificial intelligence by 15 to 40 percent [S1]. GenAI will have a significant impact across all industry sectors, with banking, high tech, and life sciences seeing the biggest impact as a percentage of their revenues

## 🔟 Evaluation Output

In [38]:
evaluation

{'overall_score': 7.5,
 'accuracy': {'score': 8,
  'notes': 'Most claims are supported by the provided summaries, but some sentences lack citations.'},
 'completeness': {'score': 6,
  'notes': 'Some planned dimensions, such as Ethical and Societal Implications and Economic and Labor Market Shifts, are not fully addressed.'},
 'citation_quality': {'score': 8,
  'notes': 'Most declarative sentences are properly cited, but some citations are missing.'},
 'structure': {'score': 7,
  'notes': 'The report has a logical flow, but some sections could be more balanced.'},
 'limitations': ['Lack of coverage of some planned dimensions',
  'Some sentences lack citations'],
 'confidence_level': 'medium'}

## 1️⃣1️⃣ PDF Path

In [39]:
pdf_path

'report_1769703545.pdf'